# **MaskedLM**

These models can be used for:
1. Fill Mask
2. Embeddings

## **Fill Mask Walkthrough**

### **Step 1: Identify the Model Class**

In [6]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("FacebookAI/roberta-base")

print(config)

RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.3.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}



### **Step 2: Import the Required Class**

In [7]:
# import the required classes
from transformers import AutoModelForMaskedLM, AutoTokenizer

### **Step 3: Load the Tokenizer and Model**

In [8]:
# Load tokenizer
roberta_tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path="FacebookAI/roberta-base",
    cache_dir=".models/facebook/roberta-base",
)

print("Vocabulary Size:", roberta_tokenizer.vocab_size)
print("Masked Token ID:", roberta_tokenizer.mask_token_id)
print("Masked Token:", roberta_tokenizer.decode(roberta_tokenizer.mask_token_id))

Vocabulary Size: 50265
Masked Token ID: 50264
Masked Token: <mask>


In [9]:
# Load model
roberta_model = AutoModelForMaskedLM.from_pretrained(
    pretrained_model_name_or_path="FacebookAI/roberta-base",
    cache_dir=".models/facebook/roberta-base",
    dtype="auto",    
    use_safetensors=True     # Forces to load .safetensors
)

print("Vocab and Embedding Size:", roberta_model.roberta.embeddings.word_embeddings)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vocab and Embedding Size: Embedding(50265, 768, padding_idx=1)


### **Step 4: Exploring the model architecture**

In [10]:
# You can print the model to take a look at its architecture

roberta_model

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): 

### **Step 5: Pass Input to Model**

In [11]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# devide = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='mps')

In [12]:
prompt = "The capital of France is <mask>."

input_ids_map = roberta_tokenizer(prompt, return_tensors="pt")

input_ids_map = input_ids_map.to(device)
roberta_model = roberta_model.to(device)

input_ids_map

{'input_ids': tensor([[    0,   133,   812,     9,  1470,    16, 50264,     4,     2]],
       device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}

In [13]:
output_ids = roberta_model(
    **input_ids_map,
)

output_ids

MaskedLMOutput(loss=None, logits=tensor([[[34.8743, -3.8884, 18.9567,  ...,  2.8639,  5.2812, 11.2557],
         [ 8.5224, -2.9420, 19.8858,  ...,  2.7513,  4.0812,  8.5191],
         [-3.2755, -4.2489,  9.6843,  ..., -1.3403, -2.0838, -0.4035],
         ...,
         [-4.8167, -3.8880,  8.3117,  ..., -4.3991, -5.6916,  1.1575],
         [20.6685, -4.4709, 20.3602,  ...,  0.9624,  3.1282,  7.1574],
         [11.6269, -3.6408, 32.1565,  ...,  2.0802, -0.2959,  9.1577]]],
       device='mps:0', grad_fn=<LinearBackward0>), hidden_states=None, attentions=None)

In [14]:
print(output_ids.logits.shape)

torch.Size([1, 9, 50265])


### **Step 6 - Extract Logits of Mask Position**

In [16]:
# Identify the Mask Index
mask_index = (input_ids_map["input_ids"] == roberta_tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

print(mask_index)

tensor([6], device='mps:0')


In [20]:
mask_logits = output_ids.logits[0, mask_index, :]

print(mask_logits.shape)

torch.Size([1, 50265])


### **Step 7 - Convert logits to Prediction**

In [21]:
predicted_token_id = torch.argmax(mask_logits, dim=-1)

print(predicted_token_id)

tensor([2201], device='mps:0')


### **Step 8 - Decode**

In [23]:
predicted_token = roberta_tokenizer.decode(predicted_token_id)

print(predicted_token)

 Paris


## **Embedding Walkthrough**

Embeddings come from Hidden State, not from logits (AKA softmax).

### **Step 1-5: Same Like Fill Mask but with AutoModel Class**

In [32]:
from transformers import AutoModel, AutoTokenizer
import torch

# Load tokenizer
roberta_tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path="FacebookAI/roberta-base",
    cache_dir=".models/facebook/roberta-base",
)

print("Vocabulary Size:", roberta_tokenizer.vocab_size)
print("Masked Token ID:", roberta_tokenizer.mask_token_id)
print("Masked Token:", roberta_tokenizer.decode(roberta_tokenizer.mask_token_id))

# Load model
roberta_model = AutoModel.from_pretrained(
    pretrained_model_name_or_path="FacebookAI/roberta-base",
    cache_dir=".models/facebook/roberta-base",
    dtype="auto",    
    use_safetensors=True     # Forces to load .safetensors
)

print("Vocab and Embedding Size:", roberta_model.embeddings.word_embeddings)

Vocabulary Size: 50265
Masked Token ID: 50264
Masked Token: <mask>


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Vocab and Embedding Size: Embedding(50265, 768, padding_idx=1)


In [27]:
roberta_model

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropou

In [33]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# devide = torch.device("cuda" if torch.cuda.is_available() else "cpu")

prompt = "The capital of France is Paris."

input_ids_map = roberta_tokenizer(prompt, return_tensors="pt")

input_ids_map = input_ids_map.to(device)
roberta_model = roberta_model.to(device)

print(input_ids_map)

outputs = roberta_model(**input_ids_map)

{'input_ids': tensor([[   0,  133,  812,    9, 1470,   16, 2201,    4,    2]],
       device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}


### **Step 6: Get Hidden State**

In [36]:
hidden_states = outputs.last_hidden_state

print(hidden_states.shape)
# Creates Raw Contextual Embedding for each token

torch.Size([1, 9, 768])


In [38]:
cls_head = outputs.pooler_output

print(cls_head.shape)
# A processed summary of the entire sequence

torch.Size([1, 768])


### **Important**

- Avoid pooler_output as it is not trained well.
- In production, use last_hidden_state + pooling/aggregation strategy
- Best Industry practices for the pooling strategy is Mean Pooling. There are other pooling techniques like Max Pooling, but rarely used in NLP.

### **Step 7: Mean Pooling**

In [39]:
import torch

# Apply attention mask
attention_mask = input_ids_map["attention_mask"]
mask = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()

# Mean Pooling
embeddings = torch.sum(hidden_states * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

# Normalize
embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

print(embeddings.shape)
# Output: [batch, hidden_dim]

torch.Size([1, 768])
